<table style="width: 100%; border-style: none;">
<tr style="border-style: none">
<td style="border-style: none; width: 1%; text-align: left; font-size: 16px">Institut f&uuml;r Theoretische Physik<br /> Universit&auml;t zu K&ouml;ln</td>
<td style="border-style: none; width: 1%; font-size: 16px">&nbsp;</td>
<td style="border-style: none; width: 1%; text-align: right; font-size: 16px">Prof. Dr. Matteo Rizzi

</tr>
</table>
<hr>
<h1 style="font-weight:bold; text-align: center; margin: 0px; font-size: 30px; padding:0px;">Computerphysik</h1>
<h1 style="font-weight:bold; text-align: center; margin: 0px; font-size: 30px; padding:0px;">Übungsblatt 6</h1>
<hr>
<h3 style="font-weight:bold; text-align: center; margin: 0px; font-size: 20px; padding:0px; margin-bottom: 20px;">Sommersemester 2026</h3>
<hr>

<table style="border-style: none; width: 100%"><tr style="border-style: none;">
<td style="border-style: none; width: 1%; text-align: left; font-size: 25px; font-weight: bold; text-decoration: underline">Aufgaben auf Übungsblatt 7</td>
<td style="border-style: none; width: 1%; text-align: right; font-size: 15px"></td></tr></table>

- Aufgabe 07_01_A
- Aufgabe 07_01_B
- **Aufgabe 07_02**

## Aufgabe 2) Die schwingende Saite

---

In dieser Aufgabe sollen Sie die Wellengleichung einer eindimensionalen schwingenden Saite numerisch untersuchen:

$$
    \partial_t^2 u(t,x) = c^2 \partial_x^2 u(t,x) \, .\quad (1)
$$

Dabei beschreibt $u(t,x)$ die Auslenkung der Saite aus der Ruhelage $u=0$, $\partial_t$ bzw. $\partial_x$ stehen für die partiellen Ableitungen in Zeit bzw. Raum und `c` ist die Ausbreitungsgeschwindigkeit der Welle.

Da dies eine zeitabhängige partielle Differentialgleichung ist, benötigt man sowohl die Anfangsbedingungen

$$
    \begin{matrix}
    u(0,x) &= u_0(x) \quad &(I1)\\ 
    \partial_t u(0,x) &= v_0(x) \quad &(I2)
    \end{matrix}
$$

als auch Randbedingungen, welche in unterschiedlicher Form auftreten können

$$
    \begin{matrix}
        u(t,0) = f_L(t)                            & \land & u(t,L) = f_R(x) \quad                            & \text{(Dirichlet)} \\
        \partial_x u(t,0) = q_L(t)                 & \land & \partial_x u(t,L) = q_R(t) \quad                 & \text{(von Neumann)} \\
        \partial_t u(t,0) - c\partial_x u(t,0) = 0 & \land & \partial_t u(t,L) + c\partial_x u(t,L) = 0 \quad & \text{(open boundary conditions)}
    \end{matrix}
$$

als auch jegliche gemischte Formen.

### a) Diskretisierung der Differentialgleichung

---

Bestimmen Sie mithilfe der Diskretisierung der zweifachen Ableitung:

$$
    \left(\partial^2_x g\right)|_x \rightarrow \frac{g(x+\epsilon) - 2g(x) + g(x-\epsilon)}{\epsilon^2}
$$

eine Iterationsvorschrift

$$
    u^{n+1}_j = f(u^n_j, u^{n-1}_j)
$$

welche die Lösung zur Zeit $t_{n}$ und $t_{n-1}$ in die Lösung zur Zeit $t_{n+1}$ transformiert. Dabei ist $u^n_j := u((n-1)\delta t, (j-1)\delta x)$ und $\delta t$ und $\delta x$ die Diskretisierungsschritte in Zeit und Raum.

Vervollständigen Sie für Dirichlet-Randbedingungen die folgende `timestep`-Funktion, welche das vorherige und jetzige Auslenkprofil ($u_0$, $u_1$) das nächste Auslenkprofil berechnet. Setzen Sie dafür $c = 1$. Die Funktion bekommt außerdem die Diskretisierungsschritte $\delta t$ und $\delta x$ übergeben sowie die Werte an den Rändern $x_1$ und $x_N$ für den aktuellen Zeitschritt.

In [1]:
# Funktion zum updaten von der aktuellen Auslenkung u_1 zum nächsten Schritt.
# Gibt den aktuellen Schritt und den nächsten Schritt zurueck
function timestep(u_0, u_1, δt, δx, u_left, u_right)
    N = length(u_1)
    κ = δt/δx   # Courant-Zahl (c = 1)

    u_next = similar(u_1)

    # Diskretisierte Wellengleichung im Inneren (j = 2,...,N-1):
    for j in 2:N-1
        u_next[j] = 2*u_1[j] - u_0[j] + κ^2*(u_1[j+1] - 2*u_1[j] + u_1[j-1])
    end

    # Dirichlet-Randbedingungen an beiden Enden:
    u_next[1]   = u_left
    u_next[end] = u_right

    return u_1, u_next
end


timestep (generic function with 1 method)

**Herleitung:** Setzt man in $(1)$ die zentrale Diskretisierung beider zweiter Ableitungen ein,

$$
\frac{u^{n+1}_j - 2u^n_j + u^{n-1}_j}{\delta t^2} = c^2 \frac{u^n_{j+1} - 2u^n_j + u^n_{j-1}}{\delta x^2},
$$

und löst nach $u^{n+1}_j$ auf, erhält man mit der Courant-Zahl $\kappa := c\,\delta t/\delta x$ (hier $c=1$) die explizite Iterationsvorschrift

$$
u^{n+1}_j = 2u^n_j - u^{n-1}_j + \kappa^2\left(u^n_{j+1} - 2u^n_j + u^n_{j-1}\right).
$$

Diese wird im Code für alle inneren Gitterpunkte $j=2,\dots,N-1$ ausgewertet; an den Rändern $j=1$ und $j=N$ werden stattdessen direkt die vorgegebenen Dirichlet-Werte `u_left` bzw. `u_right` eingesetzt. Die Funktion gibt `(u_1, u_next)` zurück, damit man im nächsten Aufruf einfach wieder `(u_0, u_1) = (u_1, u_next)` weiterschieben kann.

---

Nachdem die DGL diskretisiert wurde, müssen auch die Randbedingungen und Anfangsbedingungen noch diskretisiert und implementiert werden.

Während die Anfangsbedingung $(I1)$ und die Dirichlet-Randbedingung (DR) einfach numerisch zu implementieren sind, muss $(I2)$ auch noch diskretisiert werden.
Dafür eignet sich am besten die Vorschrift:
$$
    \partial_t u(t,x) \rightarrow \frac{1}{2\delta t}\left(u^{n+1}_j - u^{n-1}_j \right) 
$$
Für $t=0$ ($n=1$) führt auf eine Bestimmungsgleichung für $u^0_j$, welche nur noch von $u^2_j$ abhängig ist. Da $u^0_j$ einem Zeitpunkt vor unserer Simulation entspricht, kann man diese Bestimmungsgleichung
dann nutzen, um $u^0_j$ aus der Iterationsvorschrift für $u^2_j$ zu entfernen, was dann zu einer modifizierten Form
$u^2_j = \tilde{f}(u^1_j)$ führt, welche nur von der Anfangskonfiguration $u^1_j$ abhängig ist.

Die Angabe von $u^1_j$ und $u^2_j$ zusammen mit den Randbedingungen für alle Zeiten komplettiert die numerische Prozedur.

> **Aufgabe**: 
> - Bestimmen Sie $\tilde{f}$
> - Erstellen Sie eine Initialisierungsfunktion, um $u^1_j$ und $u^2_j$ mit einem vorgegebenen Startprofil und Geschwindigkeitsprofil zu initialisieren.

Setzen Sie dafür $L=1$.

**Hinweis**: 
Die Zahl $\kappa:=c\delta t/\delta x$ wird auch Courant Zahl genannt, und numerische Stabilität ist garantiert für $\kappa \le 1$

In [2]:
# Funktion, welche aus einer Funktion für die Anfangsform und eine Funktion für die Anfangsgeschwindigkeit
# u^1_j und u^2_j erzeugt.
function initialize(shape_fn, velocity_profile, δt, δx)
    x = 0:δx:1
    N = length(x)
    κ = δt/δx

    u_0 = shape_fn.(x)          # u^1_j = u(t=0, x_j),        entspricht (I1)
    v_0 = velocity_profile.(x)  # v_0(x_j),                    entspricht (I2)

    u_1 = similar(collect(u_0)) # u^2_j = u(t=δt, x_j)

    # f̃: Startschritt unter Verwendung von (I2), siehe Herleitung oben
    for j in 2:N-1
        u_1[j] = u_0[j] + δt*v_0[j] + (κ^2/2)*(u_0[j+1] - 2*u_0[j] + u_0[j-1])
    end

    # Ränder: zum Zeitpunkt δt haben sich Randwerte (bei den hier betrachteten Startprofilen) noch
    # nicht geändert, daher übernehmen wir die Randwerte des Startprofils.
    u_1[1]   = u_0[1]
    u_1[end] = u_0[end]

    return u_0, u_1, x
end;


**Herleitung von $\tilde f$:** Die zentrale Diskretisierung der Zeitableitung,
$$
\partial_t u(t,x) \rightarrow \frac{1}{2\delta t}\left(u^{n+1}_j - u^{n-1}_j\right),
$$
liefert für $n=1$ (also $t=0$) die Bestimmungsgleichung
$$
v_0(x_j) = \frac{u^2_j - u^0_j}{2\delta t} \quad\Longrightarrow\quad u^0_j = u^2_j - 2\delta t\, v_0(x_j).
$$

Setzt man dies in die Iterationsvorschrift aus a) für $n=1$ ein,
$$
u^2_j = 2u^1_j - u^0_j + \kappa^2\left(u^1_{j+1}-2u^1_j+u^1_{j-1}\right),
$$
so verschwindet der unphysikalische Punkt $u^0_j$ und man erhält nach Auflösen nach $u^2_j$:

$$
u^2_j = \tilde f(u^1_j) = u^1_j + \delta t\, v_0(x_j) + \frac{\kappa^2}{2}\left(u^1_{j+1} - 2u^1_j + u^1_{j-1}\right).
$$

Genau diese Formel wird oben für die inneren Punkte verwendet, um aus dem Startprofil `u_0` ($=u^1_j$) das Profil `u_1` ($=u^2_j$) zu berechnen.

### b) Visualisierung der Lösung

---

Die Funktion `animate` nimmt nun Ihre Startkonfiguration `u_0` und `u_1` zusammen mit der Orts- und Zeitdiskretisierung $\delta x$ und $\delta t$ entgegen, zusammen mit 
der finalen Zeit, bis zu der das System simuliert werden soll. Die Zeitevolution wird dazu mit Ihrer zuvor definierten `timestep`-Funktion durchgeführt.

Optional können Sie noch die Dirichlet-Randbedingungen als Funktion an `animate` übergeben:
```
my_bc(t) = sin(t)
anim = animate(u_0,u_1, δx,δt, T; bc_left = my_bc)
```
würde z. B. am linken Rand eine oszillierende Funktion als Randbedingung besitzen.
Die Animation kann dann durch
```
gif(anim, "ex1.gif", fps = 15)
```
erzeugt werden.


Implementieren Sie nun verschiedene Startprofile und Randbedingungen und spielen Sie ein wenig herum. Mögliche Kombinationen:

$$
    \begin{cases}
        a) & u_0(x) = 0,                       &  v_0(x) = 0,                   & \mathrm{bc}_\mathrm{right}(t) = 0, \quad \mathrm{bc}_\mathrm{left} = A\sin(\omega t)\exp(-\mu t) \\
        b) & u_0(x) = exp(-(x-x0)^2/\sigma^2), & v_0(x) = 0,                    & \mathrm{bc}_\mathrm{right}(t) = \mathrm{bc}_\mathrm{left}(t) = 0 \\
        c) & u_0(x) = 0,                       & v_0 = exp(-(x-x0)^2/\sigma^2), & \mathrm{bc}_\mathrm{right}(t) = \mathrm{bc}_\mathrm{left}(t) = 0
    \end{cases}
$$



In [3]:
function animate(u_0,u_1, δx,δt, T; bc_left = t->0, bc_right = t ->0)
    default(label = false)
    nt = Int(T/δt) + 1
    #δx = x[2]-x[1]
    x = 0:δx:1
    u_current = copy(u_1)
    u_prev    = copy(u_0)
    
    n_save = 10
    
    anim = @animate for jj in 1:n_save:nt
        if(jj == 1)
            plot(x, u_prev, xlim = (0, 1), ylim = (-1,1), markershape = :circle, color = :black)
        elseif(jj == 2)
            plot(x, u_current,  xlim = (0, 1), ylim = (-1,1), markershape = :circle, color = :black)
        else
            for kk = jj:jj+n_save-1
                u_prev, u_current = timestep(u_prev, u_current, δt, δx, bc_left(kk*δt), bc_right(kk*δt))
            end
            plot(x, u_current, xlim = (0, 1), ylim = (-1,1), markershape = :circle, color = :black)
        end
    end
    
    return anim
end

LoadError: LoadError: UndefVarError: `@animate` not defined in `Main`
Suggestion: check for spelling errors or missing imports.
in expression starting at /Users/torbenbredenbach/Documents/UNI KÖLN/Computerphysik/Übung 7/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X13sZmlsZQ==.jl:11

**Erklärung:** In allen drei Szenarien wird zunächst mit `initialize` das Anfangsprofil `u^1_j` (Startform) und `u^2_j` (mithilfe der Anfangsgeschwindigkeit) berechnet. Danach übernimmt `animate` die Zeitentwicklung mittels `timestep` und erzeugt daraus eine Animation:

- **a)** zeigt, wie sich eine anfangs ruhende Saite verhält, wenn man sie am linken Rand mit einem gedämpften Sinus "anschubst" – man sieht eine Welle, die von links nach rechts läuft und am festen rechten Rand reflektiert wird.
- **b)** zeigt die freie Zeitentwicklung eines lokalisierten Gauß-Pulses ohne Anfangsgeschwindigkeit: der Puls zerfällt symmetrisch in zwei gegenläufige Wellenpakete, die an beiden Rändern reflektiert werden.
- **c)** ist das "Gegenstück" zu b): statt einer anfänglichen Auslenkung wird ein lokalisierter Geschwindigkeitsstoß vorgegeben (die Saite ist zunächst flach), was ebenfalls zu zwei gegenläufigen Wellenpaketen führt, allerdings mit anderem Vorzeichen-/Formverhalten.

Die Courant-Zahl $\kappa = \delta t/\delta x = 0{.}5 \le 1$ garantiert numerische Stabilität (siehe Hinweis oben).

In [4]:
using Plots

# Diskretisierung (Courant-Zahl κ = δt/δx = 0.5 ≤ 1  ->  stabil)
δx = 0.01
δt = 0.005
T  = 5.0

# ------------------------------------------------------------------------
# a) Ruhende Saite, die am linken Rand durch eine gedämpfte Schwingung
#    angeregt wird; rechter Rand bleibt fixiert (u = 0).
# ------------------------------------------------------------------------
shape_a(x)    = 0.0
velocity_a(x) = 0.0

A, ω, μ = 1.0, 10.0, 0.5
bc_left_a(t)  = A*sin(ω*t)*exp(-μ*t)
bc_right_a(t) = 0.0

u0_a, u1_a, x = initialize(shape_a, velocity_a, δt, δx)
anim_a = animate(u0_a, u1_a, δx, δt, T; bc_left = bc_left_a, bc_right = bc_right_a)
gif(anim_a, "saite_a.gif", fps = 15)

# ------------------------------------------------------------------------
# b) Gauß-förmige Anfangsauslenkung, Saite zunächst in Ruhe (v0 = 0),
#    beide Ränder fest eingespannt.
# ------------------------------------------------------------------------
x0, σ = 0.5, 0.05
shape_b(x)    = exp(-(x - x0)^2/σ^2)
velocity_b(x) = 0.0

u0_b, u1_b, x = initialize(shape_b, velocity_b, δt, δx)
anim_b = animate(u0_b, u1_b, δx, δt, T)
gif(anim_b, "saite_b.gif", fps = 15)

# ------------------------------------------------------------------------
# c) Flache Saite (u0 = 0) mit einem gaußförmigen Geschwindigkeitspuls
#    als Anfangsbedingung, beide Ränder fest eingespannt.
# ------------------------------------------------------------------------
shape_c(x)    = 0.0
velocity_c(x) = exp(-(x - x0)^2/σ^2)

u0_c, u1_c, x = initialize(shape_c, velocity_c, δt, δx)
anim_c = animate(u0_c, u1_c, δx, δt, T)
gif(anim_c, "saite_c.gif", fps = 15)


MethodError: MethodError: no method matching animate(::Vector{Float64}, ::Vector{Float64}, ::Float64, ::Float64, ::Float64; bc_left::typeof(bc_left_a), bc_right::typeof(bc_right_a))
The function `animate` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  animate(::Any, ::Any; every, fps, loop, variable_palette, verbose, show_msg, kw...)
   @ Plots ~/.julia/packages/Plots/GIume/src/animation.jl:58
  animate(!Matched::Plots.FrameIterator, ::Any; kw...)
   @ Plots ~/.julia/packages/Plots/GIume/src/animation.jl:46
  animate(::Any; ...)
   @ Plots ~/.julia/packages/Plots/GIume/src/animation.jl:58
  ...


### c) Bonus Implementierung von OBC

---

Um auch andere Randbedingungen zu implementieren, müssen auch diese diskretisiert werden. In dieser Bonusaufgabe sollen nun die Dirichlet-Randbedingungen auf der rechten Seite durch offene Randbedingungen ersetzt werden.

Die offenen Randbedingungen am rechten Rand werden durch:

$$
    \partial_t u(t,L) = - c\partial_x u(t,L)
$$

bestimmt und verknüpfen die räumlichen mit den zeitlichen Ableitungen.
Fällt Ihnen eine schlaue Möglichkeit ein, die OBC Randbedingungen so zu diskretisieren, dass man eine gesonderte Iterationsvorschrift für $u^{n+1}_L$ erhält? Tipp: Diese darf nur von den vorherigen Zeitschritten abhängen.


Modifizieren Sie damit die `timestep`- und `animation`-Methode um offene Randbedingungen zu simulieren.

**Erklärung:** Anders als bei Dirichlet-Randbedingungen kennen wir am rechten Rand keinen vorgegebenen Wert `u_right(t)` mehr – stattdessen bestimmt die Bewegungsgleichung selbst, wie sich der Rand verhält. Die Diskretisierung oben liefert eine explizite Formel für `u_next[end]`, die nur von bereits bekannten Werten abhängt (genau wie im Inneren), sodass sich `timestep_obc` genauso einfach auswerten lässt wie das ursprüngliche `timestep`.

Physikalisch beschreibt die offene Randbedingung eine "unendlich lange" Fortsetzung der Saite nach rechts: eine ankommende Welle wird (näherungsweise) nicht reflektiert, sondern läuft ungestört über den Rand hinaus. Im Demo-Beispiel sieht man das gut, wenn man den Gauß-Puls aus Teilaufgabe b) verwendet: Während bei Dirichlet-Rändern der Puls reflektiert wird, verschwindet er hier einfach am rechten Rand.

In [ ]:
# ------------------------------------------------------------------------
# Offene Randbedingung (OBC) am rechten Rand:
#     ∂_t u(t,L) = -c ∂_x u(t,L)
#
# Diskretisierung:
#   Zeitableitung  -> zentraler Differenzenquotient: (u^{n+1}_N - u^{n-1}_N)/(2δt)
#   Ortsableitung  -> rückwärtiger Differenzenquotient: (u^n_N - u^n_{N-1})/δx
#                     (rückwärts, weil es keinen Gitterpunkt rechts von x_N gibt)
#
# => u^{n+1}_N = u^{n-1}_N - 2κ (u^n_N - u^n_{N-1}),   κ = δt/δx  (c = 1)
#
# Diese Vorschrift hängt nur von bereits berechneten ("vorherigen") Zeitschritten ab
# und lässt sich daher explizit auswerten.
# ------------------------------------------------------------------------

function timestep_obc(u_0, u_1, δt, δx, u_left)
    N = length(u_1)
    κ = δt/δx
    u_next = similar(u_1)

    for j in 2:N-1
        u_next[j] = 2*u_1[j] - u_0[j] + κ^2*(u_1[j+1] - 2*u_1[j] + u_1[j-1])
    end

    u_next[1]   = u_left
    u_next[end] = u_0[end] - 2*κ*(u_1[end] - u_1[end-1])   # OBC am rechten Rand

    return u_1, u_next
end

function animate_obc(u_0, u_1, δx, δt, T; bc_left = t -> 0)
    default(label = false)
    nt = Int(T/δt) + 1
    x = 0:δx:1
    u_current = copy(u_1)
    u_prev    = copy(u_0)

    n_save = 10

    anim = @animate for jj in 1:n_save:nt
        if(jj == 1)
            plot(x, u_prev, xlim = (0, 1), ylim = (-1,1), markershape = :circle, color = :black)
        elseif(jj == 2)
            plot(x, u_current,  xlim = (0, 1), ylim = (-1,1), markershape = :circle, color = :black)
        else
            for kk = jj:jj+n_save-1
                u_prev, u_current = timestep_obc(u_prev, u_current, δt, δx, bc_left(kk*δt))
            end
            plot(x, u_current, xlim = (0, 1), ylim = (-1,1), markershape = :circle, color = :black)
        end
    end

    return anim
end

# Demo: Gauß-Puls läuft nach rechts aus der Saite heraus, ohne am rechten Rand reflektiert zu werden.
u0_obc, u1_obc, x = initialize(shape_b, velocity_b, δt, δx)
anim_obc = animate_obc(u0_obc, u1_obc, δx, δt, T)
gif(anim_obc, "saite_obc.gif", fps = 15)
